# Basic Usage: Synthetic Digital Rock Demo

This tutorial demonstrates the lightweight porosity-control workflow without requiring raw micro-CT data or trained checkpoints. It loads or creates a synthetic 64^3 probability volume, applies quantile-based binarization, checks the achieved porosity, computes the mean two-point correlation curve `S2_R`, and displays middle slices.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "tutorials":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.sampling.quantile_binarization import quantile_binarize
from src.metrics.porosity import compute_porosity
from src.metrics.two_point_correlation import two_point_correlation_xyz


def smooth_probability_field(noise, sigma=2):
    try:
        from scipy.ndimage import gaussian_filter

        prob = gaussian_filter(noise, sigma=sigma)
    except ImportError:
        prob = noise.astype(np.float32, copy=True)
        iterations = max(1, int(4 * sigma))
        for _ in range(iterations):
            prob = (
                prob
                + np.roll(prob, 1, axis=0)
                + np.roll(prob, -1, axis=0)
                + np.roll(prob, 1, axis=1)
                + np.roll(prob, -1, axis=1)
                + np.roll(prob, 1, axis=2)
                + np.roll(prob, -1, axis=2)
            ) / 7.0

    return ((prob - prob.min()) / (prob.max() - prob.min())).astype(np.float32)

## 1. Load or Create a Synthetic Probability Volume

If `examples/synthetic_64.npy` is present, the tutorial uses it. Otherwise it creates a smooth random probability volume.

In [ ]:
example_path = ROOT / "examples" / "synthetic_64.npy"

if example_path.exists():
    prob = np.load(example_path).astype(np.float32)
else:
    rng = np.random.default_rng(0)
    noise = rng.random((64, 64, 64))
    prob = smooth_probability_field(noise, sigma=2)

prob.shape, float(prob.min()), float(prob.max())

## 2. Match a Target Porosity with Quantile Binarization

In [ ]:
target_porosity = 0.13
seg, threshold, achieved_porosity = quantile_binarize(prob, target_porosity, seed=0)
check_porosity = compute_porosity(seg)

print(f"Target porosity:   {target_porosity:.6f}")
print(f"Achieved porosity: {achieved_porosity:.6f}")
print(f"Check porosity:    {check_porosity:.6f}")
print(f"Threshold:         {threshold:.6f}")

## 3. Compute the Mean Two-Point Correlation Curve

In [ ]:
curves = two_point_correlation_xyz(seg, max_lag=32)
r = np.arange(curves["R"].size)

plt.figure(figsize=(5, 3.5))
plt.plot(r, curves["R"], label="S2_R")
plt.xlabel("Lag r (voxels)")
plt.ylabel("Two-point correlation")
plt.grid(alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()

## 4. Display Middle Slices

In [ ]:
mid = prob.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(prob[:, :, mid], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Probability field")
axes[0].axis("off")

axes[1].imshow(seg[:, :, mid], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Binarized sample")
axes[1].axis("off")

plt.tight_layout()

## 5. Save Tutorial Outputs

In [ ]:
out_dir = ROOT / "examples" / "demo_output"
out_dir.mkdir(parents=True, exist_ok=True)

np.save(ROOT / "examples" / "demo_input.npy", prob.astype(np.float32))
np.save(out_dir / "demo_seg.npy", seg.astype(np.uint8))

print(f"Saved probability volume to: {ROOT / 'examples' / 'demo_input.npy'}")
print(f"Saved binary volume to:      {out_dir / 'demo_seg.npy'}")